# Notebook 5: Reporte de Calidad y Conclusiones
**Autores:** Leonardo Figueroa, Miguel Flores, Steven Villegas.

**Pipeline ETL - NYC Taxi Trip Records**

**Objetivo:** Generar reporte final con métricas de calidad, tabla de optimizaciones Spark y conclusiones del proyecto.


## Configuración Inicial


In [1]:
import sys, os, json, yaml, logging, pandas as pd
from pathlib import Path

# --- MEMORIA ---
os.environ['PYSPARK_SUBMIT_ARGS'] = '--driver-memory 4g --executor-memory 4g pyspark-shell'
# ---------------

sys.path.append(os.path.abspath('..'))
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('etl')
from pyspark.sql import SparkSession, functions as F
from sqlalchemy import create_engine, text

_config_path = os.path.join(os.path.abspath('..'), "config", "etl_config.yaml")
with open(_config_path, "r", encoding="utf-8") as _f:
    _cfg = yaml.safe_load(_f)
_spark_cfg = _cfg.get("spark", {})
builder = SparkSession.builder \
    .appName("NYC_Taxi_ETL_05") \
    .master(_spark_cfg.get("master", "local[*]"))
for k, v in _spark_cfg.get("config", {}).items():
    builder = builder.config(k, v)
spark = builder.getOrCreate()

from src.config_loader import *


---
## Reporte de Calidad
**Objetivo:** Visualizar las métricas de calidad de todos los servicios y meses procesados.


In [2]:
qm_path = os.path.join(METADATA_PATH, "quality_metrics_summary")
qr_path = os.path.join(METADATA_PATH, "quality_rejected_records")
if os.path.exists(qm_path):
    df_qm = spark.read.parquet(qm_path)
    df_qm_pd = df_qm.toPandas()
    print("\n=== MÉTRICAS DE CALIDAD POR SERVICIO/MES ===")
    display(df_qm_pd)
    
    total_valid = df_qm_pd['valid_records'].sum()
    total_rejected = df_qm_pd['rejected_records'].sum()
    total_records = df_qm_pd['total_records'].sum()
    overall_quality = round((total_valid / total_records) * 100, 2) if total_records > 0 else 0
    print(f"\nCalidad global: {overall_quality}% ({total_valid} válidos / {total_records} totales)")
    print(f"Registros sospechosos: {total_rejected}")
    
    print("\n--- Resumen por servicio ---")
    for svc in df_qm_pd['service_type'].unique():
        svc_df = df_qm_pd[df_qm_pd['service_type'] == svc]
        print(f"  {svc}: {svc_df['total_records'].sum()} registros, calidad {svc_df['quality_percentage'].mean():.2f}%")
else:
    logger.warning("Quality metrics not found - run notebook 03 first")



=== MÉTRICAS DE CALIDAD POR SERVICIO/MES ===


,duplicate_records,month,null_critical_records,process_id,processed_at,quality_percentage,rejected_records,service_type,suspicious_records,total_records,valid_records,year
0,0,10,42617260,055285cd25a5,2026-06-17T22:32:53.022978,99.96,8401,fhvhv,8401,21308630,21300229,2025
1,0,11,41636374,055285cd25a5,2026-06-17T22:33:06.052764,99.90,21420,fhvhv,21420,20818187,20796767,2025
2,0,12,44216722,055285cd25a5,2026-06-17T22:33:40.199566,99.94,12597,fhvhv,12597,22108361,22095764,2025
3,0,1,41880612,055285cd25a5,2026-06-17T22:33:53.285927,99.82,37563,fhvhv,37563,20940306,20902743,2026
4,0,2,39751220,055285cd25a5,2026-06-17T22:34:08.333138,99.94,10962,fhvhv,10962,19875610,19864648,2026
...,...,...,...,...,...,...,...,...,...,...,...,...
79,0,5,0,055285cd25a5,2026-06-17T22:24:11.623577,92.93,3918,green,3918,55399,51481,2025
80,0,6,0,055285cd25a5,2026-06-17T22:24:13.110953,94.15,2889,green,2889,49390,46501,2025
81,0,7,0,055285cd25a5,2026-06-17T22:24:14.537223,95.54,2148,green,2148,48205,46057,2025
82,0,8,0,055285cd25a5,2026-06-17T22:24:16.158208,95.53,2069,green,2069,46306,44237,2025



Calidad global: 98.92% (665887486 válidos / 673147407 totales)
Registros sospechosos: 7259921

--- Resumen por servicio ---
  fhvhv: 566928980 registros, calidad 99.95%
  yellow: 104800743 registros, calidad 93.65%
  green: 1417684 registros, calidad 94.22%


In [3]:
if os.path.exists(qr_path):
    df_qr = spark.read.parquet(qr_path)
    df_qr_pd = df_qr.toPandas()
    print("\n=== REGISTROS RECHAZADOS POR CALIDAD ===")
    print(f"Total rechazos: {len(df_qr_pd)}")
    if len(df_qr_pd) > 0:
        print("\nDistribución por regla de rechazo:")
        print(df_qr_pd['rejection_rule'].value_counts().to_string())
        print("\nMuestra de rechazos:")
        display(df_qr_pd.head(10))
else:
    logger.info("No quality rejected records found")



=== REGISTROS RECHAZADOS POR CALIDAD ===
Total rechazos: 2593

Distribución por regla de rechazo:
rejection_rule
duration > 480        420
tip > 100%            420
fare_amount < 0       420
duration <= 0         342
speed > 100 mph       280
trip_distance <= 0    280
total_amount <= 0     280
pickup > dropoff      149
future pickup           2

Muestra de rechazos:


,business_reason,original_value,process_id,rejected_at,rejection_column,rejection_rule,rejection_stage,service_type,source_file,technical_reason,trip_id
0,Data does not meet quality standards,581.6666666666666,055285cd25a5,2026-06-17T22:25:47.477642,trip_duration_minutes,duration > 480,phase5_quality,fhvhv,fhvhv_tripdata_2024-04.parquet,Rule 'duration > 480' violated on column trip_...,1bcff67f23d956249d8a07f0a67794f7986e1ba87f12da...
1,Data does not meet quality standards,566.3333333333334,055285cd25a5,2026-06-17T22:25:47.477644,trip_duration_minutes,duration > 480,phase5_quality,fhvhv,fhvhv_tripdata_2024-04.parquet,Rule 'duration > 480' violated on column trip_...,2dd1ecffd1adb62040048d058e9de79bf9cf1a03274ac3...
2,Data does not meet quality standards,692.6666666666666,055285cd25a5,2026-06-17T22:25:47.477647,trip_duration_minutes,duration > 480,phase5_quality,fhvhv,fhvhv_tripdata_2024-04.parquet,Rule 'duration > 480' violated on column trip_...,40c64431e53739ad43a9fac737987a17a7ae4ce22058d3...
3,Data does not meet quality standards,548.0666666666667,055285cd25a5,2026-06-17T22:25:47.477649,trip_duration_minutes,duration > 480,phase5_quality,fhvhv,fhvhv_tripdata_2024-04.parquet,Rule 'duration > 480' violated on column trip_...,6d922024ffecdd013906747e485db23c2b6e3a74822941...
4,Data does not meet quality standards,162.51354279523292,055285cd25a5,2026-06-17T22:25:48.050042,tip_percentage,tip > 100%,phase5_quality,fhvhv,fhvhv_tripdata_2024-04.parquet,Rule 'tip > 100%' violated on column tip_perce...,003aadda256dfb4e0920f0485355c72342816d51a13c1b...
5,Data does not meet quality standards,134.89208633093526,055285cd25a5,2026-06-17T22:25:48.050049,tip_percentage,tip > 100%,phase5_quality,fhvhv,fhvhv_tripdata_2024-04.parquet,Rule 'tip > 100%' violated on column tip_perce...,005818974eed913c6d907ceffe29d811e4acab1c76a93e...
6,Data does not meet quality standards,244.79804161566707,055285cd25a5,2026-06-17T22:25:48.050052,tip_percentage,tip > 100%,phase5_quality,fhvhv,fhvhv_tripdata_2024-04.parquet,Rule 'tip > 100%' violated on column tip_perce...,010f3489d2bc9379732f348d5f617487326f20126ba3e6...
7,Data does not meet quality standards,111.79429849077698,055285cd25a5,2026-06-17T22:25:48.050057,tip_percentage,tip > 100%,phase5_quality,fhvhv,fhvhv_tripdata_2024-04.parquet,Rule 'tip > 100%' violated on column tip_perce...,014583285040549c9f0c3e906bb9f603044296f67798d8...
8,Data does not meet quality standards,113.55893708834886,055285cd25a5,2026-06-17T22:25:48.050059,tip_percentage,tip > 100%,phase5_quality,fhvhv,fhvhv_tripdata_2024-04.parquet,Rule 'tip > 100%' violated on column tip_perce...,01b1cb04a8a1f58ad7b288f2ea71da7921f35ecd84be7a...
9,Data does not meet quality standards,-1.66,055285cd25a5,2026-06-17T22:25:59.845480,fare_amount,fare_amount < 0,phase5_quality,fhvhv,fhvhv_tripdata_2024-05.parquet,Rule 'fare_amount < 0' violated on column fare...,04486624a853d1ad88f0068844d697182e32759a3e4a1c...


---
## Verificación en Base de Datos
**Objetivo:** Conectarse a SQLite y verificar las tablas cargadas.


In [4]:
db_path = DB_PATH
if os.path.exists(db_path):
    engine = create_engine(f"sqlite:///{db_path}", echo=False)
    with engine.connect() as conn:
        tables = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table'")).fetchall()
        print("\n=== TABLAS EN SQLITE ===")
        for t in tables:
            count = conn.execute(text(f"SELECT COUNT(*) FROM [{t[0]}]")).fetchone()[0]
            print(f"  {t[0]}: {count} registros")
        
        print("\n=== Q1 - Revenue por servicio ===")
        result = conn.execute(text("""
            SELECT service_type, COUNT(*) AS total_trips, SUM(total_amount) AS total_revenue
            FROM gold_trips_clean GROUP BY service_type ORDER BY total_revenue DESC
        """))
        print(pd.DataFrame(result.fetchall(), columns=result.keys()).to_string(index=False))
        
        print("\n=== Q2 - Métricas de calidad ===")
        result = conn.execute(text("""
            SELECT service_type, year, month, total_records, valid_records,
                   rejected_records, quality_percentage
            FROM quality_metrics_summary ORDER BY year, month, service_type
        """))
        print(pd.DataFrame(result.fetchall(), columns=result.keys()).to_string(index=False))
else:
    logger.warning("SQLite DB not found - run notebook 04 first")



=== TABLAS EN SQLITE ===
  gold_daily_revenue: 2564 registros
  gold_location_performance: 149335 registros
  quality_rejected_records: 2593 registros
  quality_metrics_summary: 84 registros
  audit_file_inventory: 84 registros
  gold_trips_clean: 3371576 registros

=== Q1 - Revenue por servicio ===
service_type  total_trips  total_revenue
      yellow      3371576    98928891.74

=== Q2 - Métricas de calidad ===
service_type  year  month  total_records  valid_records  rejected_records  quality_percentage
       fhvhv  2024      1       19663879       19657231              6648               99.97
       green  2024      1          56551          53103              3448               93.90
      yellow  2024      1        2964624        2864053            100571               96.61
       fhvhv  2024      2       19359093       19344469             14624               99.92
       green  2024      2          53577          50234              3343               93.76
      yellow  2024

---
## Optimizaciones Spark Aplicadas
**Objetivo:** Documentar las optimizaciones implementadas y su justificación.


In [5]:
optimizations = [
    ["1", "Adaptive Query Execution (AQE)", "Optimiza joins y agregaciones según datos reales",
     "spark.sql.adaptive.enabled=true"],
    ["2", "Coalesce de Particiones Dinámico", "Reduce número de particiones en shuffle",
     "spark.sql.adaptive.coalescePartitions.enabled=true"],
    ["3", "Vectorized Parquet Reader", "Lectura de Parquet hasta 10x más rápida",
     "spark.sql.parquet.enableVectorizedReader=true"],
    ["4", "Partition Overwrite Mode dinámico", "Sobrescribe solo particiones necesarias",
     "spark.sql.sources.partitionOverwriteMode=dynamic"],
    ["5", "Zona horaria UTC", "Evita problemas de husos horarios en timestamps",
     "spark.sql.session.timeZone=UTC"],
    ["6", "Particionamiento por año/mes", "Filtrado eficiente y poda de particiones",
     "year y month como columnas de partición en Bronze/Silver"],
    ["7", "Deduplicación por trip_id (SHA-256)", "Elimina duplicados con Window + row_number",
     "Window.partitionBy(trip_id).orderBy(ingestion_timestamp)"],
    ["8", "Escritura en modo overwrite por partición", "Idempotencia en cada ejecución",
     "write.mode('overwrite').parquet(...)"]
]
print("\n=== OPTIMIZACIONES SPARK ===")
print(f"{'#':<3} {'Técnica':<40} {'Justificación':<55} {'Configuración':<45}")
print("-"*143)
for opt in optimizations:
    print(f"{opt[0]:<3} {opt[1]:<40} {opt[2]:<55} {opt[3]:<45}")
print("-"*143)



=== OPTIMIZACIONES SPARK ===
#   Técnica                                  Justificación                                           Configuración                                
-----------------------------------------------------------------------------------------------------------------------------------------------
1   Adaptive Query Execution (AQE)           Optimiza joins y agregaciones según datos reales        spark.sql.adaptive.enabled=true              
2   Coalesce de Particiones Dinámico         Reduce número de particiones en shuffle                 spark.sql.adaptive.coalescePartitions.enabled=true
3   Vectorized Parquet Reader                Lectura de Parquet hasta 10x más rápida                 spark.sql.parquet.enableVectorizedReader=true
4   Partition Overwrite Mode dinámico        Sobrescribe solo particiones necesarias                 spark.sql.sources.partitionOverwriteMode=dynamic
5   Zona horaria UTC                         Evita problemas de husos horarios en t

---
## Conclusiones
**Objetivo:** Resumir los hallazgos y resultados del pipeline ETL.


In [6]:
print("""
=== CONCLUSIONES DEL PIPELINE ETL - NYC TAXI TRIP RECORDS ===
1. ARQUITECTURA MEDALLÓN:
   Se implementó la arquitectura de 4 capas (Raw → Bronze → Silver → Gold)
   permitiendo trazabilidad y reprocesamiento incremental.
2. HOMOLOGACIÓN DE ESQUEMAS:
   Se unificaron 3 esquemas diferentes (yellow, green, fhvhv) bajo un esquema
   canónico de 24 columnas, manejando columnas faltantes con valores nulos.
3. CALIDAD DE DATOS:
   Se aplicaron 9 reglas de negocio para detectar viajes sospechosos,
   permitiendo filtrar datos anómalos antes del análisis.
4. DETECCIÓN DE DUPLICADOS:
   Se generó un trip_id único mediante SHA-256 y se eliminaron duplicados
   usando Window + row_number.
5. MANEJO DE ARCHIVOS CORRUPOS:
   Se implementó cuarentena con clasificación de errores (7 categorías)
   y procesamiento de archivos del repositorio parquet-testing.
6. PERSISTENCIA:
   Los datos Gold se cargaron en SQLite con 6 tablas analíticas listas
   para consultas y visualización.
""")



=== CONCLUSIONES DEL PIPELINE ETL - NYC TAXI TRIP RECORDS ===
1. ARQUITECTURA MEDALLÓN:
   Se implementó la arquitectura de 4 capas (Raw → Bronze → Silver → Gold)
   permitiendo trazabilidad y reprocesamiento incremental.
2. HOMOLOGACIÓN DE ESQUEMAS:
   Se unificaron 3 esquemas diferentes (yellow, green, fhvhv) bajo un esquema
   canónico de 24 columnas, manejando columnas faltantes con valores nulos.
3. CALIDAD DE DATOS:
   Se aplicaron 9 reglas de negocio para detectar viajes sospechosos,
   permitiendo filtrar datos anómalos antes del análisis.
4. DETECCIÓN DE DUPLICADOS:
   Se generó un trip_id único mediante SHA-256 y se eliminaron duplicados
   usando Window + row_number.
5. MANEJO DE ARCHIVOS CORRUPOS:
   Se implementó cuarentena con clasificación de errores (7 categorías)
   y procesamiento de archivos del repositorio parquet-testing.
6. PERSISTENCIA:
   Los datos Gold se cargaron en SQLite con 6 tablas analíticas listas
   para consultas y visualización.



---
## Fin del Pipeline ETL
